# QLoRA Fine-Tuning Falcon-7B for Customer Support — Reference Notebook

> **Reference notebook.** See [`fine_tuning.md`](../06-07-08-transfer-learning-fine-tuning/fine_tuning.md)
> for the LoRA/QLoRA concepts behind every config value used here, and [`README.md`](./README.md) for
> the fuller walkthrough this notebook distills.

**Methods covered:**
- Loading a causal LM in 4-bit (`BitsAndBytesConfig`) with CPU offload for int8 ops
- Preparing a quantized model for training with `prepare_model_for_kbit_training` (freezing base
  weights, upcasting norms/output head, enabling gradient checkpointing)
- Attaching LoRA adapters (`LoraConfig` + `get_peft_model`) and inspecting the trainable-parameter ratio
- Formatting question/answer pairs into a single completion-style string and fine-tuning with the plain
  `Trainer` + `DataCollatorForLanguageModeling(mlm=False)` (causal LM objective)
- Evaluating generated answers with **BLEU**
- Running inference with mixed-precision autocast

**Use this as a reference when:** you need copy-paste-ready code for QLoRA fine-tuning a causal LM on a
small completion-style Q&A dataset, including the modern `prepare_model_for_kbit_training` setup path.

**Don't use this as a reference for:** `SFTTrainer`-based fine-tuning (see the QLoRA notebook in
[module 06-07-08](../06-07-08-transfer-learning-fine-tuning/qlora_sentiment_finetuning.ipynb)) or
seq2seq fine-tuning (see [module 09](../09-legal-assistant-llm-finetuning/flan_t5_legal_qa_finetuning.ipynb)).

In [ ]:
import json
import torch
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# 4-bit NF4 storage with double quantization for extra memory savings; fp32 CPU offload lets
# layers that don't fit on the GPU spill over instead of failing to load.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

In [ ]:
base_repo = "tiiuae/falcon-7b"

model = AutoModelForCausalLM.from_pretrained(base_repo, quantization_config=quantization_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(base_repo)

In [ ]:
# Replaces the manual freeze-params / upcast-norms / enable-gradient-checkpointing / cast-lm-head
# dance with a single, current PEFT call that does all of it consistently.
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)

In [ ]:
# Sanity check that QLoRA is working as intended: only a tiny fraction of the 7B parameters
# (the adapters) should be trainable.
def print_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable_params} || all params: {all_params} || trainable%: {100 * trainable_params / all_params:.4f}")

print_trainable_parameters(model)

In [ ]:
with open("dataset.json") as f:
    raw_data = json.load(f)

questions = [item["pergunta"] for item in raw_data["perguntas"]]
answers = [item["resposta"] for item in raw_data["perguntas"]]

dataset = Dataset.from_dict({"question": questions, "answer": answers})
dataset = dataset.train_test_split(test_size=0.15)

In [ ]:
# Completion-style format: question, an explicit "->:" marker, then the answer -- the entire
# merged string is tokenized and used as-is, so the causal LM objective computes loss over the
# whole sequence (question tokens included), not just the answer.
def merge_columns(example):
    example["text"] = example["question"] + " ->: " + example["answer"]
    return example

dataset = dataset.map(merge_columns)
dataset = dataset.map(lambda batch: tokenizer(batch["text"]), batched=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model.config.use_cache = False  # incompatible with gradient checkpointing

trainer = Trainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=TrainingArguments(
        eval_strategy="epoch",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=10,          # more epochs than usual -- dataset is only a handful of examples
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="outputs",
        report_to="none",
    ),
    # mlm=False selects causal (next-token) language modeling, matching Falcon's decoder-only architecture.
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

In [ ]:
def predict(question):
    model.eval()
    device = next(model.parameters()).device

    batch = tokenizer(f"{question} ->: ", return_tensors="pt", padding=True, truncation=True)
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad(), torch.cuda.amp.autocast():
        output_tokens = model.generate(**batch, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)

    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)

predictions = [predict(q) for q in dataset["test"]["question"]]

In [ ]:
bleu = evaluate.load("bleu")
references = dataset["test"]["text"]

result = bleu.compute(predictions=predictions, references=references)
print(result)

In [ ]:
question = "Como posso criar uma conta?"
device = next(model.parameters()).device

inputs = tokenizer(question, return_tensors="pt", padding=True, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad(), torch.cuda.amp.autocast():
    output_tokens = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(output_tokens[0], skip_special_tokens=True))